In [1]:
#Clone your repo and install dependencies

import os
os.environ["DISABLE_NEPTUNE"] = "1"        # no Neptune account needed
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # this code uses a single GPU

%cd /kaggle/working
!rm -rf spai-parameterized-radius
!git clone https://github.com/Kashshaf-Labib/spai-parameterized-radius.git
%cd /kaggle/working/spai-parameterized-radius
!pip install -q -r requirements-kaggle.txt
print("SETUP DONE")


/kaggle/working
Cloning into 'spai-parameterized-radius'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 175 (delta 41), reused 171 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 35.21 MiB | 41.35 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/spai-parameterized-radius
  Preparing metadata (setup.py) ... done
SETUP DONE


In [2]:
# Confirm the GPU is on

import torch
print("torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — turn on GPU in Settings!")


torch: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


In [3]:
# Download the pretrained SPAI weights

!mkdir -p weights
!gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O weights/spai.pth
!ls -lh weights


Downloading...
From (original): https://drive.google.com/uc?id=1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI
From (redirected): https://drive.google.com/uc?id=1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI&confirm=t&uuid=df1f6ab8-80df-4d4d-9557-be1ce3f2293f
To: /kaggle/working/spai-parameterized-radius/weights/spai.pth
100%|█████████████████████████████████████████| 935M/935M [00:03<00:00, 234MB/s]
total 892M
-rw-r--r-- 1 root root 892M Apr  3  2025 spai.pth


In [4]:
# Build the dataset list (CSV)

!mkdir -p datasets
!python -m spai.tools.create_dir_csv \
  --train_dir medical_spai_dataset/train \
  --val_dir medical_spai_dataset/val \
  -o datasets/medical_smoke.csv \
  -r .

import pandas as pd
df = pd.read_csv("datasets/medical_smoke.csv")
print(df.groupby(["split","class"]).size())


split  class
train  0        20
       1        20
val    0         4
       1         4
dtype: int64


In [5]:
# Baseline run: fixed radius r=16

!python -m spai train \
  --cfg configs/spai.yaml \
  --batch-size 4 \
  --data-path datasets/medical_smoke.csv \
  --csv-root-dir . \
  --finetune-from weights/spai.pth \
  --output output/fixed_radius \
  --tag smoke \
  --amp-opt-level O0 \
  --data-workers 2 \
  --save-all \
  --opt TRAIN.EPOCHS 2 \
  --opt TRAIN.WARMUP_EPOCHS 1 \
  --opt PRINT_FREQ 5 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 \
  --opt DATA.TEST_PREFETCH_FACTOR 1


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.14). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
=> merge config from configs/spai.yaml
[2026-07-07 15:04:01 finetune](__main__.py 269): INFO Full config saved to output/fixed_radius/finetune/smoke/config.json
[2026-07-07 15:04:01 finetune](__main__.py 272): INFO AMP_OPT_LEVEL: O0
AUG:
  AUTO_AUGMENT: rand-m9-mstd0.5-inc1
  BLUR_PROB: 0.25
  COLOR_JITTER: 0.0
  COLOR_JITTER_BRIGHTNESS_RANGE: &id001
  - 0.8
  - 1.2
  COLOR_JITTER_CONTRAST_RANGE: *id001
  COLOR_JITTER_HUE_RANGE:
  - -0.1
  - 0.1
  COLOR_JITTER_SATURATION_RANGE: *id001
  CUTMIX: 1.0
  CUTMIX_MINMAX: null
  GAUSSIAN_BLUR_LIMIT:
  - 3
  - 9
  GAUSSIAN_BLUR_PROB: 0.5
  GAUSSIAN_BLUR_SIGMA:
  - 0.01
  - 0.5
  GAUSSIAN_NOISE_PROB: 0.5
  HORIZONTAL_FLIP_PROB: 0.5

In [6]:
#learnable radius

!python -m spai train \
  --cfg configs/spai_learnable_radius.yaml \
  --batch-size 4 \
  --data-path datasets/medical_smoke.csv \
  --csv-root-dir . \
  --finetune-from weights/spai.pth \
  --output output/learnable_radius \
  --tag smoke \
  --amp-opt-level O0 \
  --data-workers 2 \
  --save-all \
  --opt TRAIN.EPOCHS 5 \
  --opt TRAIN.WARMUP_EPOCHS 1 \
  --opt TRAIN.RADIUS_LR 0.05 \
  --opt PRINT_FREQ 5 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 \
  --opt DATA.TEST_PREFETCH_FACTOR 1


/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.14). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
=> merge config from configs/spai_learnable_radius.yaml
[2026-07-07 15:08:54 finetune](__main__.py 269): INFO Full config saved to output/learnable_radius/finetune/smoke/config.json
[2026-07-07 15:08:54 finetune](__main__.py 272): INFO AMP_OPT_LEVEL: O0
AUG:
  AUTO_AUGMENT: rand-m9-mstd0.5-inc1
  BLUR_PROB: 0.25
  COLOR_JITTER: 0.0
  COLOR_JITTER_BRIGHTNESS_RANGE: &id001
  - 0.8
  - 1.2
  COLOR_JITTER_CONTRAST_RANGE: *id001
  COLOR_JITTER_HUE_RANGE:
  - -0.1
  - 0.1
  COLOR_JITTER_SATURATION_RANGE: *id001
  CUTMIX: 1.0
  CUTMIX_MINMAX: null
  GAUSSIAN_BLUR_LIMIT:
  - 3
  - 9
  GAUSSIAN_BLUR_PROB: 0.5
  GAUSSIAN_BLUR_SIGMA:
  - 0.01
  - 0.5
  GAUSSIAN_NOISE_PROB: 0.5
  HORI

In [7]:
#  Read the learned radius straight from the saved checkpoints

import glob, torch
for ckpt in sorted(glob.glob("output/learnable_radius/finetune/smoke/ckpt_epoch_*.pth")):
    r = torch.load(ckpt, map_location="cpu", weights_only=False)["model"]["mfvit.soft_frequency_mask.radius"].item()
    print(f"{ckpt}:  learned radius = {r:.4f}")


output/learnable_radius/finetune/smoke/ckpt_epoch_0.pth:  learned radius = 16.0839
output/learnable_radius/finetune/smoke/ckpt_epoch_1.pth:  learned radius = 15.8785
output/learnable_radius/finetune/smoke/ckpt_epoch_2.pth:  learned radius = 15.7442
output/learnable_radius/finetune/smoke/ckpt_epoch_3.pth:  learned radius = 15.7163
output/learnable_radius/finetune/smoke/ckpt_epoch_4.pth:  learned radius = 15.7126


In [8]:
# Evaluate both models on the validation split

print("========== FIXED RADIUS (baseline) ==========")
!python -m spai test --cfg configs/spai.yaml --batch-size 1 \
  --model output/fixed_radius/finetune/smoke --output output/test_fixed --tag smoke_fixed \
  --split val --test-csv datasets/medical_smoke.csv --test-csv-root-dir . \
  --opt MODEL.PATCH_VIT.MINIMUM_PATCHES 4 --opt DATA.NUM_WORKERS 2 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 --opt DATA.TEST_PREFETCH_FACTOR 1

print("========== LEARNABLE RADIUS (ours) ==========")
!python -m spai test --cfg configs/spai_learnable_radius.yaml --batch-size 1 \
  --model output/learnable_radius/finetune/smoke --output output/test_learnable --tag smoke_learnable \
  --split val --test-csv datasets/medical_smoke.csv --test-csv-root-dir . \
  --opt MODEL.PATCH_VIT.MINIMUM_PATCHES 4 --opt DATA.NUM_WORKERS 2 \
  --opt MODEL.FEATURE_EXTRACTION_BATCH 128 --opt DATA.TEST_PREFETCH_FACTOR 1


========== FIXED RADIUS (baseline) ==========
/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.14). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
=> merge config from configs/spai.yaml
[2026-07-07 15:11:43 finetune](__main__.py 421): INFO Full config saved to output/test_fixed/finetune/smoke_fixed/config.json
[2026-07-07 15:11:43 finetune](__main__.py 424): INFO AMP_OPT_LEVEL: ''
AUG:
  AUTO_AUGMENT: rand-m9-mstd0.5-inc1
  BLUR_PROB: 0.25
  COLOR_JITTER: 0.0
  COLOR_JITTER_BRIGHTNESS_RANGE: &id001
  - 0.8
  - 1.2
  COLOR_JITTER_CONTRAST_RANGE: *id001
  COLOR_JITTER_HUE_RANGE:
  - -0.1
  - 0.1
  COLOR_JITTER_SATURATION_RANGE: *id001
  CUTMIX: 1.0
  CUTMIX_MINMAX: null
  GAUSSIAN_BLUR_LIMIT:
  - 3
  - 9
  GAUSSIAN_BLUR_PROB: 0.5
  GAUSSIAN_BLUR_SIGMA:
  - 0.01
  - 0.5
  GA